# 🎬 Movie Recommendation System
### SVD (Matrix Factorization) · MovieLens 100K
---

## What is SVD?
SVD = Singular Value Decomposition.
It looks at all the ratings and learns hidden patterns.
Example: It figures out 'this user likes action movies' without you telling it.
Then it uses those patterns to predict ratings for movies the user hasn't seen yet.

It's pure math (linear algebra) — no deep learning involved.

## Step 1 — Install & Import

In [ ]:
# Run this once to install
# !pip install scikit-surprise pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import Dataset, SVD, accuracy
from surprise.model_selection import train_test_split

plt.style.use('seaborn-v0_8-darkgrid')
print('✅ Libraries imported!')

## Step 2 — Load Dataset

In [ ]:
# MovieLens 100K is built into the surprise library
# It auto-downloads — no manual download needed
data = Dataset.load_builtin('ml-100k')

# Convert to DataFrame so we can explore it
df = pd.DataFrame(data.raw_ratings, columns=['user_id', 'movie_id', 'rating', 'timestamp'])
df['user_id']  = df['user_id'].astype(int)
df['movie_id'] = df['movie_id'].astype(int)
df['rating']   = df['rating'].astype(float)

print(f'Total ratings  : {len(df):,}')
print(f'Unique users   : {df["user_id"].nunique()}')
print(f'Unique movies  : {df["movie_id"].nunique()}')
df.head()

## Step 3 — Explore the Data (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('MovieLens 100K — Data Overview', fontsize=14, fontweight='bold')

# Plot 1 — Rating Distribution
rc = df['rating'].value_counts().sort_index()
axes[0].bar(rc.index, rc.values, color='steelblue', edgecolor='white', width=0.6)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating (1–5)')
axes[0].set_ylabel('Count')

# Plot 2 — Ratings per User
rpu = df.groupby('user_id')['rating'].count()
axes[1].hist(rpu, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of Ratings Given')
axes[1].axvline(rpu.mean(), color='darkred', linestyle='--', label=f'Mean: {rpu.mean():.0f}')
axes[1].legend()

# Plot 3 — Ratings per Movie
rpm = df.groupby('movie_id')['rating'].count()
axes[2].hist(rpm, bins=40, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Ratings per Movie')
axes[2].set_xlabel('Number of Ratings Received')
axes[2].axvline(rpm.mean(), color='darkgreen', linestyle='--', label=f'Mean: {rpm.mean():.0f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('eda.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA plots saved as eda.png')

## Step 4 — Sparsity
Most user-movie pairs have NO rating. This is the core challenge — predicting the missing ones.

In [ ]:
total_possible = df['user_id'].nunique() * df['movie_id'].nunique()
sparsity = 1 - (len(df) / total_possible)

print(f'Total possible ratings : {total_possible:,}')
print(f'Actual ratings         : {len(df):,}')
print(f'Sparsity               : {sparsity * 100:.2f}%')
print()
print('→ 93% of user-movie pairs are unknown. SVD predicts these!')

## Step 5 — Train SVD Model

In [ ]:
# Split data: 80% training, 20% testing
trainset, testset = train_test_split(data, test_size=0.20, random_state=42)

# Train SVD
# n_factors = number of hidden features to learn (e.g. action-level, romance-level)
# n_epochs  = how many times it goes through the data to improve
model = SVD(n_factors=100, n_epochs=30, lr_all=0.005, reg_all=0.02, random_state=42)
model.fit(trainset)

print('✅ SVD model trained!')

## Step 6 — Evaluate the Model
RMSE and MAE tell us how accurate our predictions are.
If RMSE = 0.93, our rating predictions are off by less than 1 star on average.

In [ ]:
predictions = model.test(testset)

rmse = accuracy.rmse(predictions)
mae  = accuracy.mae(predictions)

print(f'\nRMSE : {rmse:.4f}  (how far off our predictions are on average)')
print(f'MAE  : {mae:.4f}  (mean absolute error)')

## Step 7 — Get Top 10 Recommendations for a User

In [ ]:
def recommend(user_id, n=10):
    # Movies this user has already seen
    already_rated = set(df[df['user_id'] == user_id]['movie_id'].tolist())

    # All movies in dataset
    all_movies = df['movie_id'].unique().tolist()

    # Predict rating for every movie the user hasn't seen
    preds = [
        (movie_id, model.predict(str(user_id), str(movie_id)).est)
        for movie_id in all_movies
        if movie_id not in already_rated
    ]

    # Sort by predicted rating — highest first
    preds.sort(key=lambda x: x[1], reverse=True)

    # Print top N
    print(f'\n🎬 Top {n} Recommendations for User {user_id}:')
    print(f'{"Rank":<6} {"Movie ID":<12} {"Predicted Rating"}')
    print('-' * 35)
    for rank, (mid, rating) in enumerate(preds[:n], 1):
        stars = '⭐' * round(rating)
        print(f'{rank:<6} {mid:<12} {rating:.2f}  {stars}')


# Try for different users
recommend(user_id=1)
recommend(user_id=50)

## Step 8 — Visualise Prediction Accuracy

In [ ]:
actuals    = [p.r_ui for p in predictions]
predicted  = [p.est  for p in predictions]
errors     = [abs(a - p) for a, p in zip(actuals, predicted)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SVD Model — Prediction Analysis', fontsize=13, fontweight='bold')

# Actual vs Predicted scatter
axes[0].scatter(actuals, predicted, alpha=0.2, color='steelblue', s=5)
axes[0].plot([1, 5], [1, 5], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual Rating')
axes[0].set_ylabel('Predicted Rating')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

# Error distribution
axes[1].hist(errors, bins=40, color='coral', edgecolor='white')
axes[1].axvline(np.mean(errors), color='darkred', linestyle='--', label=f'Mean error: {np.mean(errors):.2f}')
axes[1].set_xlabel('Absolute Error')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Evaluation plots saved as model_evaluation.png')

---
## ✅ Project Complete!

| Step | What we did |
|------|-------------|
| 1 | Loaded MovieLens 100K dataset |
| 2 | Explored data with visualisations |
| 3 | Understood the sparsity problem |
| 4 | Trained SVD to learn hidden user/movie patterns |
| 5 | Evaluated with RMSE and MAE |
| 6 | Generated top-N recommendations for any user |

**RMSE ~0.93 — less than 1 star error on average. This is how Amazon recommends at scale!**